# Train langmuir 2d cnn regressor

In [1]:
from __future__ import annotations
import argparse
import contextlib
import copy
import json
import random
from dataclasses import asdict, dataclass
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from scipy import stats
from torch import nn
from torch.utils.data import DataLoader, Dataset
from stochastic_fano import load_stochastic_fano, sample_spectral_images
from concentration_split import grouped_concentration_split, regression_metrics


In [2]:
PACKAGE = Path.cwd().parent
ROOT = PROJECT = PROJECT_ROOT = PACKAGE / 'work'
SETTINGS = argparse.Namespace(data_dir=PACKAGE / 'work/langmuir_spectral_images_double_ellipse_S843_theta205_D6p52_random10000_logC_m10p5_m8_t7200_dt60_allparam_noise0p05_spectrum0p01', out_dir=PACKAGE / 'work/models/double_ellipse_S843_theta205_D6p52_random10000_logC_m10p5_m8_t7200_dt60_allparam_noise0p05_spectrum0p01', figure_dir=PACKAGE / 'work/figures/double_ellipse_S843_theta205_D6p52_random10000_logC_m10p5_m8_t7200_dt60_allparam_noise0p05_spectrum0p01', image_height=0, image_width=0, normalization='minmax', split_seed=7, train_fraction=0.7, val_fraction=0.15, test_fraction=0.15, seeds='0', epochs=80, batch_size=4, learning_rate=0.0001, weight_decay=1e-06, optimizer='nadam', patience=50, base_channels=8, dropout=0.0, norm='group', loss='mse', smoothl1_beta=0.05, pool_height=4, pool_width=3, conv_kernel_height=3, conv_kernel_width=3, pool_schedule='symmetric', max_normalizer_images=512, num_workers=0, resample_training_noise=True, training_dtype='float16')


In [3]:
@dataclass
class ImageNormalizer:
    method: str
    mean: float | None = None
    std: float | None = None
    min_value: float | None = None
    max_value: float | None = None

    def apply(self, image: np.ndarray) -> np.ndarray:
        image = image.astype(np.float32, copy=False)
        if self.method == 'none':
            return image
        if self.method == 'standard':
            return (image - self.mean) / self.std
        if self.method == 'minmax':
            return (image - self.min_value) / (self.max_value - self.min_value)


In [4]:
@dataclass
class SplitIndices:
    train: np.ndarray
    val: np.ndarray
    test: np.ndarray

In [5]:
class SpectralImageDataset(Dataset):

    def __init__(self, images: np.ndarray, metadata: pd.DataFrame, indices: np.ndarray, normalizer: ImageNormalizer, image_size: tuple[int, int] | None, tensor_dtype: torch.dtype) -> None:
        self.images = images
        self.metadata = metadata.reset_index(drop=True)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.normalizer = normalizer
        self.image_size = image_size
        self.tensor_dtype = tensor_dtype
        concentration = self.metadata['concentrationM'].to_numpy(dtype=np.float32)
        self.target_log10 = np.log10(concentration).astype(np.float32)

    def __len__(self) -> int:
        return len(self.indices)

    def __getitem__(self, item: int) -> tuple[torch.Tensor, torch.Tensor]:
        index = int(self.indices[item])
        image = self.normalizer.apply(self.get_image(index))
        x = torch.from_numpy(image[None, :, :].copy()).to(self.tensor_dtype)
        if self.image_size is not None and x.shape[-2:] != self.image_size:
            x = F.interpolate(x[None], size=self.image_size, mode='bilinear', align_corners=False).squeeze(0)
        y = torch.tensor([self.target_log10[index]], dtype=torch.float32)
        return (x, y)

    def get_image(self, index: int) -> np.ndarray:
        return np.asarray(self.images[index], dtype=np.float32)


In [6]:
class ResampledSpectralImageDataset(SpectralImageDataset):
    """Monte Carlo acquisition noise with a reproducible record/epoch seed."""

    def configure(self, data_dir: Path, seed: int) -> None:
        self.noise_config = json.loads((data_dir / 'spectral_image_database_summary.json').read_text())
        self.fano = load_stochastic_fano(Path(self.noise_config['fano_dir']), self.noise_config['wavelength_stride'])
        self.thickness = np.load(data_dir / 'thickness_nm.npy', mmap_mode='r')
        self.seed = seed
        self.epoch = 0

    def set_epoch(self, epoch: int) -> None:
        self.epoch = epoch

    def get_image(self, index: int) -> np.ndarray:
        streams = np.random.SeedSequence([918271, self.seed, self.epoch, index]).spawn(2)
        image = sample_spectral_images(self.thickness[index:index + 1], self.fano, np.random.default_rng(streams[0]))[0]
        std = self.noise_config['spectrum_noise_std']
        if std > 0:
            image += np.random.default_rng(streams[1]).normal(0, std, image.shape).astype(np.float32)
            image = np.clip(image, self.noise_config['clip_min'], self.noise_config['clip_max'])
        return image.astype(self.images.dtype).astype(np.float32)


In [7]:
class ConvStage(nn.Module):

    def __init__(self, in_channels: int, out_channels: int, norm: str, kernel_size: tuple[int, int]=(3, 3), pool_size: tuple[int, int]=(2, 2)) -> None:
        super().__init__()
        padding = tuple((size // 2 for size in kernel_size))
        self.stage = nn.Sequential(nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, padding=padding), make_normalization(norm, out_channels), nn.ReLU(inplace=True), nn.Conv2d(out_channels, out_channels, kernel_size=kernel_size, padding=padding), make_normalization(norm, out_channels), nn.ReLU(inplace=True), nn.MaxPool2d(kernel_size=pool_size))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.stage(x)


In [8]:
def make_normalization(norm: str, channels: int) -> nn.Module:
    if norm == 'batch':
        return nn.BatchNorm2d(channels)
    if norm == 'group':
        n_groups = min(8, channels)
        while channels % n_groups != 0:
            n_groups -= 1
        return nn.GroupNorm(n_groups, channels)
    if norm == 'none':
        return nn.Identity()


In [9]:
class SpectralImageCNNRegressor(nn.Module):

    def __init__(self, base_channels: int=16, dropout: float=0.15, norm: str='batch', pooled_size: tuple[int, int]=(8, 6), image_shape: tuple[int, int]=(351, 51), conv_kernel: tuple[int, int]=(3, 3), pool_schedule: tuple[tuple[int, int], ...]=((2, 2), (2, 2), (2, 2), (2, 2))) -> None:
        super().__init__()
        _ = image_shape
        c = base_channels
        pooled_features = 8 * c * pooled_size[0] * pooled_size[1]
        self.features = nn.Sequential(ConvStage(1, c, norm, conv_kernel, pool_schedule[0]), ConvStage(c, 2 * c, norm, conv_kernel, pool_schedule[1]), ConvStage(2 * c, 4 * c, norm, conv_kernel, pool_schedule[2]), ConvStage(4 * c, 8 * c, norm, conv_kernel, pool_schedule[3]))
        self.feature_dim = pooled_features
        self.regressor = nn.Sequential(nn.AdaptiveAvgPool2d(pooled_size), nn.Flatten(), nn.Linear(pooled_features, 4 * c), nn.ReLU(inplace=True), nn.Dropout(p=dropout), nn.Linear(4 * c, 1))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        features = self.features(x)
        features = self.regressor[:-1](features)
        with torch.autocast(device_type=x.device.type, enabled=False):
            return self.regressor[-1](features.float())


In [10]:
def make_pool_schedule(name: str) -> tuple[tuple[int, int], ...]:
    if name == 'symmetric':
        return ((2, 2), (2, 2), (2, 2), (2, 2))
    if name == 'preserve-time':
        return ((2, 1), (2, 1), (2, 2), (2, 2))

In [11]:
def save_training_history(history: list[dict], seed_dir: Path) -> None:
    pd.DataFrame(history).to_csv(seed_dir / 'history.csv', index=False)

In [12]:
@torch.no_grad()
def evaluate_loss(model: nn.Module, loader: DataLoader, criterion, device, compute_dtype: torch.dtype) -> float:
    model.eval()
    total = 0.0
    for xb, yb in loader:
        xb, yb = (xb.to(device), yb.to(device))
        with autocast_context(device, compute_dtype):
            prediction = model(xb)
        total += criterion(prediction.float(), yb.float()).item() * xb.size(0)
    return total / len(loader.dataset)


In [13]:
@torch.no_grad()
def predict(model: nn.Module, loader: DataLoader, device, compute_dtype: torch.dtype) -> np.ndarray:
    model.eval()
    predictions = []
    for xb, _yb in loader:
        with autocast_context(device, compute_dtype):
            prediction = model(xb.to(device))
        predictions.append(prediction.float().cpu().numpy().reshape(-1))
    return np.concatenate(predictions)


In [14]:
def autocast_context(device: torch.device, compute_dtype: torch.dtype):
    if compute_dtype == torch.float32:
        return contextlib.nullcontext()
    return torch.autocast(device_type=device.type, dtype=compute_dtype)


In [15]:
def parse_training_dtype(name: str) -> torch.dtype:
    if name == 'float32':
        return torch.float32
    if name == 'float16':
        return torch.float16


In [16]:
def make_loss(args: argparse.Namespace) -> nn.Module:
    if args.loss == 'mse':
        return nn.MSELoss()
    if args.loss == 'smoothl1':
        return nn.SmoothL1Loss(beta=args.smoothl1_beta)


In [17]:
def make_optimizer(args: argparse.Namespace, model: nn.Module):
    if args.optimizer == 'adam':
        return torch.optim.Adam(model.parameters(), lr=args.learning_rate, weight_decay=args.weight_decay)
    if args.optimizer == 'adamw':
        return torch.optim.AdamW(model.parameters(), lr=args.learning_rate, weight_decay=args.weight_decay)
    if args.optimizer == 'nadam':
        return torch.optim.NAdam(model.parameters(), lr=args.learning_rate, weight_decay=args.weight_decay)


In [18]:
def parse_image_size(args: argparse.Namespace, image_shape: tuple[int, ...]) -> tuple[int, int] | None:
    if args.image_height == 0 and args.image_width == 0:
        return None
    if (args.image_height, args.image_width) != tuple(image_shape[1:3]):
        print(f'Warning: resizing spectral images from {tuple(image_shape[1:3])} to {(args.image_height, args.image_width)}. For physical wavelength/time data, prefer changing the builder strides when possible.')
    return (args.image_height, args.image_width)


In [19]:
def fit_image_normalizer(images: np.ndarray, train_indices: np.ndarray, method: str, max_images: int, seed: int) -> ImageNormalizer:
    if method == 'none':
        return ImageNormalizer(method='none')
    rng = np.random.default_rng(seed)
    sample = train_indices
    if len(sample) > max_images:
        sample = rng.choice(sample, size=max_images, replace=False)
    values = np.asarray(images[sample], dtype=np.float32)
    if method == 'standard':
        std = float(values.std())
        return ImageNormalizer(method=method, mean=float(values.mean()), std=std)
    if method == 'minmax':
        min_value = float(values.min())
        max_value = float(values.max())
        return ImageNormalizer(method=method, min_value=min_value, max_value=max_value)


In [20]:
def parse_seeds(seed_string: str) -> list[int]:
    return [int(seed.strip()) for seed in seed_string.split(',') if seed.strip()]


In [21]:
def set_global_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


In [22]:
def serializable_args(args: argparse.Namespace) -> dict:
    values = vars(args).copy()
    for key, value in values.items():
        if isinstance(value, Path):
            values[key] = str(value)
    return values


In [23]:
args = SETTINGS

args.out_dir.mkdir(parents=True, exist_ok=True)

args.figure_dir.mkdir(parents=True, exist_ok=True)

set_global_seed(args.split_seed)

images = np.load(args.data_dir / 'spectral_images.npy', mmap_mode='r')

metadata = pd.read_csv(args.data_dir / 'spectral_image_metadata.csv')

splits = grouped_concentration_split(metadata, args)

normalizer = fit_image_normalizer(images, splits.train, args.normalization, args.max_normalizer_images, args.split_seed)

seed_metrics = []


In [ ]:
for seed in parse_seeds(args.seeds):
    print(f'\n=== Seed {seed} ===')
    seed_dir = args.out_dir / f'seed_{seed:03d}'
    seed_fig_dir = args.figure_dir / f'seed_{seed:03d}'
    seed_dir.mkdir(parents=True, exist_ok=True)
    seed_fig_dir.mkdir(parents=True, exist_ok=True)
    figure_dir = seed_fig_dir

    set_global_seed(seed)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    tensor_dtype = parse_training_dtype(args.training_dtype)
    image_size = parse_image_size(args, images.shape)
    datasets = {'train': SpectralImageDataset(images, metadata, splits.train, normalizer, image_size, tensor_dtype), 'val': SpectralImageDataset(images, metadata, splits.val, normalizer, image_size, tensor_dtype), 'test': SpectralImageDataset(images, metadata, splits.test, normalizer, image_size, tensor_dtype), 'train_eval': SpectralImageDataset(images, metadata, splits.train, normalizer, image_size, tensor_dtype)}

    if getattr(args, 'resample_training_noise', False):
        datasets['train'] = ResampledSpectralImageDataset(images, metadata, splits.train, normalizer, image_size, tensor_dtype)
        datasets['train'].configure(args.data_dir, seed)
    loaders = {name: DataLoader(dataset, batch_size=args.batch_size, shuffle=name == 'train', num_workers=args.num_workers, pin_memory=torch.cuda.is_available()) for name, dataset in datasets.items()}
    model = SpectralImageCNNRegressor(args.base_channels, args.dropout, args.norm, (args.pool_height, args.pool_width), image_size if image_size is not None else tuple(images.shape[1:3]), (args.conv_kernel_height, args.conv_kernel_width), make_pool_schedule(args.pool_schedule)).to(device)
    optimizer = make_optimizer(args, model)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)
    criterion = make_loss(args)
    scaler = torch.amp.GradScaler('cuda', enabled=device.type == 'cuda' and tensor_dtype == torch.float16)
    best_state = None
    best_val = float('inf')
    best_epoch = 0
    wait = 0
    history = []
    for epoch in range(1, args.epochs + 1):
        if args.resample_training_noise:
            loaders['train'].dataset.set_epoch(epoch)
        train_epoch_model = model
        train_epoch_loader = loaders['train']
        train_epoch_optimizer = optimizer
        train_epoch_criterion = criterion
        train_epoch_device = device
        train_epoch_compute_dtype = tensor_dtype
        train_epoch_scaler = scaler
        train_epoch_model.train()
        train_epoch_total = 0.0
        for train_epoch_xb, train_epoch_yb in train_epoch_loader:
            train_epoch_xb, train_epoch_yb = (train_epoch_xb.to(train_epoch_device), train_epoch_yb.to(train_epoch_device))
            train_epoch_optimizer.zero_grad()
            with autocast_context(train_epoch_device, train_epoch_compute_dtype):
                train_epoch_prediction = train_epoch_model(train_epoch_xb)
            train_epoch_loss = train_epoch_criterion(train_epoch_prediction.float(), train_epoch_yb.float())
            if train_epoch_scaler is not None:
                train_epoch_scaler.scale(train_epoch_loss).backward()
                train_epoch_scaler.step(train_epoch_optimizer)
                train_epoch_scaler.update()
            else:
                train_epoch_loss.backward()
                train_epoch_optimizer.step()
            train_epoch_total += train_epoch_loss.item() * train_epoch_xb.size(0)
        train_loss = train_epoch_total / len(train_epoch_loader.dataset)
        train_eval_loss = evaluate_loss(model, loaders['train_eval'], criterion, device, tensor_dtype)
        val_loss = evaluate_loss(model, loaders['val'], criterion, device, tensor_dtype)
        scheduler.step(val_loss)
        lr = optimizer.param_groups[0]['lr']
        history.append({'epoch': epoch, 'train_loss': train_loss, 'train_eval_loss': train_eval_loss, 'val_loss': val_loss, 'learning_rate': lr})
        if val_loss < best_val - 1e-07:
            best_val = val_loss
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            torch.save({'model_state_dict': best_state, 'args': serializable_args(args), 'normalization': asdict(normalizer), 'image_size': image_size, 'best_epoch': best_epoch, 'best_val_loss': best_val, 'train_indices': splits.train.tolist(), 'val_indices': splits.val.tolist(), 'test_indices': splits.test.tolist()}, seed_dir / f'best_epoch_{epoch:03d}.pt')
            wait = 0
        else:
            wait += 1
        save_training_history(history, seed_dir)
        if epoch == 1 or epoch % 10 == 0 or wait == 0:
            print(f'epoch {epoch:03d}: train_loss={train_loss:.6g} train_eval_loss={train_eval_loss:.6g} val_loss={val_loss:.6g}')
        if wait >= args.patience:
            print(f'Early stopping at epoch {epoch}; best epoch {best_epoch}.')
            break
    model.load_state_dict(best_state)
    test_loss = evaluate_loss(model, loaders['test'], criterion, device, tensor_dtype)
    predictions = predict(model, loaders['test'], device, tensor_dtype)
    test_indices = splits.test
    truth_log10 = np.log10(metadata.loc[test_indices, 'concentrationM'].to_numpy(dtype=np.float64))
    metrics = regression_metrics(truth_log10, predictions)
    save_training_history(history, seed_dir)
    pd.DataFrame({'recordId': metadata.loc[test_indices, 'recordId'].to_numpy(), 'concentrationM': metadata.loc[test_indices, 'concentrationM'].to_numpy(), 'true_log10_concentration': truth_log10, 'pred_log10_concentration': predictions, 'pred_concentrationM': np.power(10.0, predictions)}).to_csv(seed_dir / 'test_predictions.csv', index=False)
    torch.save({'model_state_dict': best_state, 'normalization': asdict(normalizer), 'image_size': image_size, 'training_dtype': args.training_dtype, 'resample_training_noise': args.resample_training_noise, 'regression_output_dtype': 'float32', 'base_channels': args.base_channels, 'dropout': args.dropout, 'norm': args.norm, 'optimizer': args.optimizer, 'loss': args.loss, 'smoothl1_beta': args.smoothl1_beta, 'pooled_size': (args.pool_height, args.pool_width), 'conv_kernel': (args.conv_kernel_height, args.conv_kernel_width), 'pool_schedule': make_pool_schedule(args.pool_schedule), 'feature_dim': model.feature_dim, 'best_epoch': best_epoch, 'best_val_loss': best_val, 'test_loss_at_best_val_epoch': test_loss, 'train_indices': splits.train.tolist(), 'val_indices': splits.val.tolist(), 'test_indices': splits.test.tolist()}, seed_dir / 'best_model.pt')


    output_path = figure_dir / 'training_validation_loss.svg'
    frame = pd.DataFrame(history)
    label = 'MSE' if args.loss == 'mse' else f'SmoothL1 beta={args.smoothl1_beta:g}'
    fig, ax = plt.subplots(figsize=(7.0, 4.6))
    ax.plot(frame['epoch'], frame['train_eval_loss'], label='Train loss (eval mode)', linewidth=1.8)
    ax.plot(frame['epoch'], frame['val_loss'], label='Validation loss', linewidth=1.8)
    ax.scatter([best_epoch], [best_val], color='crimson', s=36, zorder=4, label=f'Best val epoch {best_epoch}')
    ax.axhline(test_loss, color='0.25', linestyle='--', linewidth=1.2, label='Held-out test loss at best val')
    ax.set_title('Langmuir 2D CNN training curve')
    ax.set_xlabel('Epoch')
    ax.set_ylabel(f'{label} loss on log10 concentration')
    ax.grid(True, alpha=0.28)
    ax.legend(frameon=False)
    fig.tight_layout()
    #fig.savefig(output_path)
    #plt.close(fig)

    y_true = truth_log10
    y_pred = predictions
    output_path = figure_dir / 'prediction_vs_ground_truth.svg'
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.scatter(y_true, y_pred, s=12, alpha=0.75)
    lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
    ax.plot(lims, lims, 'k--', linewidth=1)
    ax.set_xlabel('True log10 concentration (M)')
    ax.set_ylabel('Predicted log10 concentration (M)')
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    #fig.savefig(output_path)
    #plt.close(fig)
    metrics = metrics
    seed_metrics.append({'seed': seed, **metrics})



=== Seed 0 ===


In [ ]:
summarize_metrics_seed_metrics = seed_metrics
summarize_metrics_frame = pd.DataFrame(summarize_metrics_seed_metrics)
summarize_metrics_rows = []

for summarize_metrics_metric in summarize_metrics_frame.columns:
    if summarize_metrics_metric == 'seed':
        continue
    summarize_metrics_rows.append({'metric': summarize_metrics_metric, 'mean': summarize_metrics_frame[summarize_metrics_metric].mean(), 'std': summarize_metrics_frame[summarize_metrics_metric].std(ddof=1)})

summary = pd.DataFrame(summarize_metrics_rows)
pd.DataFrame(seed_metrics).to_csv(args.out_dir / 'seed_test_metrics.csv', index=False)
summary.to_csv(args.out_dir / 'summary_metrics_mean_std.csv', index=False)

with (args.out_dir / 'experiment_summary.json').open('w', encoding='utf-8') as handle:
    json.dump({'args': serializable_args(args), 'image_shape': list(images.shape), 'normalization': asdict(normalizer), 'n_train': int(len(splits.train)), 'n_val': int(len(splits.val)), 'n_test': int(len(splits.test)), 'target': 'log10_concentrationM', 'metrics': summary.to_dict(orient='records')}, handle, indent=2)

print('\nAggregate test metrics:')
print(summary.to_string(index=False))
